In [11]:
import os
import glob
import pandas as pd
from pathlib import Path

# 1. Find where this notebook is saved
try:
    current_dir = Path(__file__).resolve().parent
except NameError:
    current_dir = Path(os.getcwd()).resolve()

# 2. Step UP out of 'notebooks' to the project root, then DOWN into 'data'
if current_dir.name == "notebooks":
    project_root = current_dir.parent
else:
    project_root = current_dir

input_folder = project_root / "data" / "interim"
output_folder = project_root / "data" / "processed"
output_filename = "merged_features.csv"

# Create the processed folder automatically if it's missing
os.makedirs(output_folder, exist_ok=True)

# 3. Grab all CSV files inside that precise folder
search_path = os.path.join(input_folder, "*.csv")
csv_files = sorted(glob.glob(search_path))
csv_files = [f for f in csv_files if os.path.basename(f) != output_filename]
base_file = str(input_folder / "additional_pair_features.csv")
if base_file not in csv_files:
    raise FileNotFoundError(
        f"Required base file not found: {base_file}"
    )
merge_files = [f for f in csv_files if f != base_file]

print(f"Project root directory: {project_root}")
print(f"Looking inside input path: {input_folder}")
print(f"Successfully found {len(csv_files)} file(s) to merge!")


Project root directory: /Users/vincentlu/26-the-deep-learners-analysis/FinalProject
Looking inside input path: /Users/vincentlu/26-the-deep-learners-analysis/FinalProject/data/interim
Successfully found 4 file(s) to merge!


In [12]:
# Use the canonical additional-pair table as the row universe.
# This prevents outer joins from creating new rows whose combined
# daily-text feature is undefined.
print(f"Reading base file: {base_file}")
master_df = pd.read_csv(base_file)
master_df.columns = master_df.columns.str.strip().str.lower()
master_df = master_df.loc[:, ~master_df.columns.str.startswith("unnamed:")]

id_names = ["user_a", "user_b"]
missing_base_keys = [key for key in id_names if key not in master_df.columns]
if missing_base_keys:
    raise KeyError(f"Base file is missing pair keys: {missing_base_keys}")
if master_df.duplicated(id_names).any():
    raise ValueError("Base file contains duplicate participant pairs")
print(f"Tracking keys detected: {id_names}")



Reading base file: /Users/vincentlu/26-the-deep-learners-analysis/FinalProject/data/interim/additional_pair_features.csv
Tracking keys detected: ['user_a', 'user_b']


In [13]:
# 3. Loop through and merge the remaining files sideways
for file in merge_files:
    print(f"Merging: {file}")
    df_next = pd.read_csv(file)
    
    # Clean up column names to prevent case or space mismatches
    df_next.columns = df_next.columns.str.strip().str.lower()
    df_next = df_next.loc[:, ~df_next.columns.str.startswith("unnamed:")]
    
    # Double check if both id_names are present in this specific file
    missing_keys = [key for key in id_names if key not in df_next.columns]
    if missing_keys:
        print(f"⚠️ Warning: Skipping {file} because it is missing tracking key columns: {missing_keys}")
        print(f"   Available columns in this file: {list(df_next.columns)}")
        continue  # Safely skip non-pair-level files

    if df_next.duplicated(id_names).any():
        raise ValueError(f"Duplicate participant pairs found in {file}")

    duplicate_features = [
        column for column in df_next.columns
        if column in master_df.columns and column not in id_names
    ]
    if duplicate_features:
        print(f"   Skipping duplicate columns: {duplicate_features}")
        df_next = df_next.drop(columns=duplicate_features)

    master_df = pd.merge(
        master_df,
        df_next,
        on=id_names,
        how="left",
        validate="one_to_one",
    )

# Rebuild the combined daily inbound-text feature from its two inputs.
# In the source feature definition, no recorded inbound texts is 0.
daily_text_columns = [
    "user_a_daily_texts_received",
    "user_b_daily_texts_received",
]
missing_daily_columns = [
    column for column in daily_text_columns
    if column not in master_df.columns
]
if missing_daily_columns:
    raise KeyError(
        f"Cannot rebuild combined daily texts; missing {missing_daily_columns}"
    )

combined_column = "combined_daily_texts_received"
missing_before = (
    int(master_df[combined_column].isna().sum())
    if combined_column in master_df.columns else len(master_df)
)
master_df[daily_text_columns] = master_df[daily_text_columns].fillna(0)
master_df[combined_column] = master_df[daily_text_columns].sum(axis=1)
assert master_df[combined_column].notna().all()

# 4. Save the wide feature matrix into the processed directory
destination_path = os.path.join(output_folder, output_filename)
master_df.to_csv(destination_path, index=False)

print(f"\nSuccess! Combined files aligned and saved to '{destination_path}'")
print(f"Total rows in final dataset: {len(master_df)}")
print(f"Missing combined daily texts corrected: {missing_before}")
print(
    "Remaining missing combined daily texts: "
    f"{master_df[combined_column].isna().sum()}"
)


Merging: /Users/vincentlu/26-the-deep-learners-analysis/FinalProject/data/interim/calls.csv
⚠️ Warning: Skipping /Users/vincentlu/26-the-deep-learners-analysis/FinalProject/data/interim/calls.csv because it is missing tracking key columns: ['user_a', 'user_b']
   Available columns in this file: ['timestamp', 'caller', 'callee', 'duration']
Merging: /Users/vincentlu/26-the-deep-learners-analysis/FinalProject/data/interim/fb_friends.csv
Merging: /Users/vincentlu/26-the-deep-learners-analysis/FinalProject/data/interim/sms.csv
⚠️ Warning: Skipping /Users/vincentlu/26-the-deep-learners-analysis/FinalProject/data/interim/sms.csv because it is missing tracking key columns: ['user_a', 'user_b']
   Available columns in this file: ['timestamp', 'sender', 'recipient']

Success! Combined files aligned and saved to '/Users/vincentlu/26-the-deep-learners-analysis/FinalProject/data/processed/merged_features.csv'
Total rows in final dataset: 82851
Missing combined daily texts corrected: 0
Remaining mi